# Datasets Example Use 

This notebook shows how to use the various PyTorch dataset classes.

In [ ]:
import os
import yaml
import numpy as np

import torch
from torchvision import transforms
from torch.utils.data import DataLoader

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

from pollen_datasets.poleno.registry import register_condition_fn

In [ ]:
def load(config_file):
    with open(config_file, 'r') as stream:
        try:
            config = yaml.safe_load(stream)
            return config
        except yaml.YAMLError as exc:
            print(exc)
    
config = load("base_ldm_config.yaml")
dataset_config = config["dataset"]
condition_config = config["conditioning"]


**Dataset Configuration**

In [ ]:
dataset_config

**Transforms**: Definition of pytorch transforms

In [ ]:
# Image transformations
transforms_list = []

transforms_list.append(transforms.ToTensor())

transforms_list.append(transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR))

transforms_list.append(transforms.Normalize([0.5], [0.5]))

transform = transforms.Compose(transforms_list)

**Condition**

In [ ]:
condition_config = {
    'enabled': 'class+regionprops',                             
    'encoders': {  

        'class': {
            'type': 'categorical',
            'encoder': 'Embedding',
            'use_columns': 'species_norm_enum',
            'params': {
                'num_classes': 80, 
                'out_dim': 256
                }
        },

        'regionprops': {
            'type': 'numeric_vector',
            'encoder': 'MLP',
            'params': {
                'in_dim': 15, 
                'hidden_dim': 128, 
                'out_dim': 256
            },
            'use_columns': [
                'area',
                'bbox_area',
                'convex_area',
                'eccentricity',
                'equivalent_diameter',
                'feret_diameter_max',
                'major_axis_length',
                'minor_axis_length',
                'max_intensity',
                'min_intensity',
                'mean_intensity',
                'orientation',
                'perimeter',
                'perimeter_crofton',
                'solidity'
            ]
        },

        'image': {
            'type': 'image',
            'encoder': 'CLIPImageEncoder',
            'params': {
                'out_dim': 256, 
                'pretrained': 'openai/clip-vit-base-patch32'
            },
            'use_columns': 'cond_img_path'
        },
        
        'rotation': {
            'type': 'angle',
            'encoder': 'Identity',
            'params': {
                'out_dim': 4
            },
            'use_columns': None,
            'condition_fn': 'relative_viewpoint_rotation'
        }
    }
}

## 1. HolographyImageFolder

In [ ]:
from pollen_datasets.poleno import HolographyImageFolder

# Dataset
dataset = HolographyImageFolder(
    root=dataset_config["root"], 
    transform=transform, 
    labels=dataset_config["labels_train"],
    dataset_cfg=dataset_config,
    cond_cfg=condition_config,
    verbose=True,
)

In [ ]:
# Dataloader
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

**Get Batch**

In [ ]:
images, condition, filepath = next(iter(dataloader))

In [ ]:
images.shape

In [ ]:
condition

In [ ]:
filepath

## 2. PairwiseHolographyImageFolder

In [ ]:
from pollen_datasets.poleno import PairwiseHolographyImageFolder

# Dataset
dataset = PairwiseHolographyImageFolder(
    root=dataset_config["root"], 
    transform=transform, 
    labels=dataset_config["labels_train"],
    dataset_cfg=dataset_config,
    cond_cfg=condition_config,
    verbose=True,
)

# Dataloader
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

# Get Batch
(img1, img2), (cond1, cond2), (filepath1, filepath2) = next(iter(dataloader))

print(f"\n{'-' * 8} {'Output'} {'-' * 8}")
print("IMAGE BATCH SHAPE:", img1.shape)
print("CONDITION:", cond1)
print("FILEPATH:", filepath1)

## 3. StackedPairwiseHolographyImageFolder

In [ ]:
config = load("config.yaml")
dataset_config = config["dataset"]
condition_config["enabled"] = "class"


In [ ]:
from pollen_datasets.poleno.datasets import StackedPairwiseHolographyImageFolder

# Dataset
dataset = StackedPairwiseHolographyImageFolder(
    root=dataset_config["root"], 
    transform=transform, 
    labels=dataset_config["labels_train"],
    dataset_cfg=dataset_config,
    cond_cfg=condition_config,
    merge_conditions=False,
    verbose=True,
    
)

# Dataloader
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

In [ ]:
images, cond, files = next(iter(dataloader))

cond

In [ ]:
images, (cond1, cond2), (filename1, filename2) = next(iter(dataloader))

In [ ]:
images.shape

In [ ]:
cond1